# SIV pathway enrichment — interactive exploration

Run `python scripts/01_analyze.py` from the repo root first to populate `results/csv/`.

This notebook loads those outputs and lets you poke at the binary matrix,
the FAU anchor, and the Mantel-like comparison without re-running the pipeline.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

CSV = Path('..') / 'results' / 'csv'

binmat   = pd.read_csv(CSV / '01_binary_pathway_x_cluster.csv', index_col=0)
pmat     = pd.read_csv(CSV / '02_padjust_pathway_x_cluster.csv', index_col=0)
fau      = pd.read_csv(CSV / '03_FAU_pathways_detailed.csv')
d_full   = pd.read_csv(CSV / '06_cluster_distances_full.csv',   index_col=0)
d_binary = pd.read_csv(CSV / '07_cluster_distances_binary.csv', index_col=0)
summary  = json.loads((CSV / '08_analysis_summary.json').read_text())

summary

## 1. Binary matrix at a glance

In [ ]:
print(f'Shape: {binmat.shape}')
print(f'Density: {binmat.values.mean():.3f}')
binmat.sum(axis=0).rename('# significant pathways').to_frame()

## 2. Mantel-like correlation — full vs binary

In [ ]:
iu = np.triu_indices(d_full.shape[0], k=1)
r  = np.corrcoef(d_full.values[iu], d_binary.values[iu])[0, 1]
print(f'Mantel-like r = {r:.3f}')

## 3. FAU anchor — top pathways by significance

In [ ]:
(fau.assign(nlogp=-np.log10(fau['p.adjust'].clip(lower=1e-300)))
    .sort_values('nlogp', ascending=False)
    [['Cluster', 'Description', 'p.adjust', 'nlogp', 'Count']]
    .head(15))

## 4. Try another anchor

Run `python scripts/01_analyze.py --gene RPL10` from the shell and rerun the
cell above with the new filename.